In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from study.data import cohort, features, prediction_metadata
from study.io import WORK, load, write_table
from study.models import fit_model, predict
from study.statistics import paired_difference, prediction_rows, risk_matrix

In [ ]:
source_fits = load(WORK / 'models/source.pkl')

metadata, x = cohort('eicu').sort_index(), features('eicu')
shared = load(WORK / 'feature_sets.pkl')['shared']
parameters = source_fits['parameters']

In [ ]:
# Fit each tier on all eligible eICU patients
rows = []
for tier in ('R1', 'R2', 'R3'):
    model = fit_model(x[shared[tier]], metadata.in_hospital_mortality, 'xgboost', parameters)

    # Score both complete MIMIC target cohorts
    for domain in ('metavision', 'mimiciv'):
        target = cohort(domain).sort_index()
        risk = predict(model, features(domain).loc[target.index])
        rows.append(
            prediction_rows(
                prediction_metadata(target),
                risk,
                domain = domain,
                tier = tier,
                condition = 'full',
                source_patients = len(metadata),
                source_hospitals = metadata.hospital_id.nunique(),
                repeat = 0,
                group = 'all',
            )
        )

In [ ]:
# Match the full MetaVision source size across eICU hospital pools
sample_size = len(source_fits['fit_ids']['metavision', 0])
sample_deaths = round(sample_size * metadata.in_hospital_mortality.mean())
census = (
    metadata.groupby('hospital_id')
    .size()
    .rename('patients')
    .reset_index()
    .sort_values(['patients', 'hospital_id'], ascending = [False, True])
)

In [ ]:
# Define narrower hospital groups with enough patients for the matched sample
groups = {}
for number, start in enumerate((1, 2, 4, 6), 1):
    ranked = census.iloc[start:]
    count = int((ranked.patients.cumsum() < sample_size).sum() + 1)
    groups['G' + str(number)] = ranked.head(count).hospital_id.tolist()

# Pair diverse and narrower hospital pools across repeated samples
repeats = source_fits['allocation_repeats']
for repeat in repeats:
    pools = [('diverse', 'all', metadata.index, repeat)] + [
        ('narrow', group, metadata.index[metadata.hospital_id.isin(sites)], repeat + len(repeats) * number)
        for number, (group, sites) in enumerate(groups.items(), 1)
    ]

    # Match the source sample size and number of deaths within each pool
    for condition, group, pool, sample_seed in pools:
        labels = metadata.loc[pool, 'in_hospital_mortality']
        rng = np.random.default_rng(sample_seed)
        sample = pd.Index(
            np.sort(
                np.r_[
                    rng.choice(pool[labels.eq(1)], sample_deaths, replace = False),
                    rng.choice(pool[labels.eq(0)], sample_size - sample_deaths, replace = False)
                ]
            )
        )

        # Fit each tier on the same sample and score both MIMIC cohorts
        for tier in ('R1', 'R2', 'R3'):
            model = fit_model(
                x.loc[sample, shared[tier]], metadata.loc[sample, 'in_hospital_mortality'], 'xgboost', parameters
            )

            for domain in ('metavision', 'mimiciv'):
                target = cohort(domain).sort_index()
                risk = predict(model, features(domain).loc[target.index])
                rows.append(
                    prediction_rows(
                        prediction_metadata(target),
                        risk,
                        domain = domain,
                        tier = tier,
                        condition = condition,
                        source_patients = len(sample),
                        source_hospitals = metadata.loc[sample, 'hospital_id'].nunique(),
                        repeat = repeat,
                        group = group,
                    )
                )

predictions = pd.concat(rows, ignore_index = True)

write_table(WORK / 'predictions/reciprocal.parquet', predictions)

In [ ]:
# Compare reverse transfer increments across source conditions
rows = []
for (domain, condition, group), frame in predictions.groupby(['domain', 'condition', 'group']):
    ids = pd.Index(sorted(frame.stay_id.unique()))

    # Pair adjacent tiers within each target and source condition
    for richer, simpler in [('R2', 'R1'), ('R3', 'R2')]:
        result = paired_difference(
            prediction_metadata(cohort(domain).loc[ids]),
            risk_matrix(frame.loc[frame.tier.eq(richer)], ids),
            risk_matrix(frame.loc[frame.tier.eq(simpler)], ids),
            cluster = False,
        )

        rows.append(
            dict(
                domain = domain,
                condition = condition,
                source_patients = int(frame.source_patients.iloc[0]),
                source_hospitals_minimum = int(frame.source_hospitals.min()),
                source_hospitals_maximum = int(frame.source_hospitals.max()),
                group = group,
                contrast = richer + '_minus_' + simpler,
                **result,
            )
        )

# Save reverse-transfer information increments
write_table(WORK / 'results/reciprocal.parquet', pd.DataFrame(rows))

display(pd.DataFrame(rows).round(5))